# REF Academic Paper Evaluator (v4)
**Copyright: Jac Ison 2025**

### Intended use
Desktop/notebook tool to assist academic reviewers in pre-assessing research papers against REF2029 criteria (originality, significance, rigour). This is **decision support, not a replacement for expert review** — see `5.3 Ethical and Institutional Implications` in the accompanying dissertation.

### Assumptions
- Each ~1500-token segment contains enough information to be meaningfully evaluated.
- The LLM's REF-style judgement is a useful but imperfect proxy for expert reviewers.
- Input papers are in English and are readable PDF or TXT.
- The REF scoring scale (1–4, mapped to REF bands) is applied consistently.

### How it works
1. Upload a PDF or TXT paper.
2. Text is extracted and tokenised, then split into ~1500-token segments.
3. A configurable number of segments are scored by an LLM (Cohere `command-a-03-2025`) on originality, significance, and rigour (1–4), using a strict, calibration-aware system prompt.
4. A calibration step compresses over-generous or internally inconsistent scores (see Design Rationale, dissertation §3.5).
5. An impact-keyword heuristic applies a small weighting factor.
6. Results are mapped to a REF band (1*–4*, with High/Middle/Low sub-bands) and cached.

### Limitations (carried over from the dissertation, §3.6 / §7)
- Does not replace human judgement.
- Textual proxies for originality/rigour may miss contribution types (datasets, software, non-textual outputs).
- Results may not generalise beyond life sciences without domain adaptation.
- Only the first N segments of long papers are scored by default (configurable — see Config cell), to balance API cost against context coverage. **You will be warned if a paper is truncated.**
- Upstream model updates can shift outputs even with deterministic temperature settings.

### What changed from v3
- **Security:** API key now loaded from a `.env` file (never hardcoded). See setup instructions below.
- **Robustness:** LLM JSON parsing now retries once on failure instead of silently zeroing a segment's scores.
- **Transparency:** the app now tells you explicitly when a paper has been truncated to the segment cap, and how many tokens/segments were skipped.
- **Maintainability:** cache keys now include a hash of the model name + prompt version, so changing the scoring logic doesn't silently serve stale cached results.
- **Code quality:** type hints, docstrings, and separation of concerns (extraction / scoring / calibration / UI) for easier testing and reuse.

## Setup (run once)

1. Create a file named `.env` in the same folder as this notebook (do **not** commit it to git — add `.env` to `.gitignore`).
2. Put your Cohere key in it as a single line:
   ```
   COHERE_API_KEY=your-actual-key-here
   ```
3. Run the cells below in order.

If you've previously used this tool with a hardcoded key in any notebook, `.env` file, or `.zip`, **rotate that key in your Cohere dashboard** — treat any key that has ever appeared in plain text in a file you've shared or uploaded anywhere as compromised.

In [ ]:
!pip install cohere PyPDF2 nltk ipywidgets python-dotenv --quiet

In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import os
import pickle
import re
import shutil
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import cohere
import ipywidgets as widgets
import nltk
from dotenv import load_dotenv
from IPython.display import display
from nltk.tokenize import word_tokenize
from PyPDF2 import PdfReader

warnings.filterwarnings("ignore", category=UserWarning)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

## Config

Everything you're likely to want to tune lives here. `MAX_SEGMENTS` is the segment cap discussed in the dissertation's Design Rationale (§3.5) — raise it for thorough (slower, more expensive) runs of long papers, or set it to `None` to score every segment.

In [ ]:
load_dotenv()

COHERE_API_KEY: Optional[str] = os.getenv("COHERE_API_KEY")
if not COHERE_API_KEY:
    raise RuntimeError(
        "COHERE_API_KEY not found. Create a .env file alongside this notebook "
        "containing: COHERE_API_KEY=your-actual-key-here"
    )

MODEL_NAME = "command-a-03-2025"
PROMPT_VERSION = "v4-strict-calibrated"  # bump this whenever you change the system prompt or scoring logic
CACHE_DIR = Path("ref_cache")
CACHE_DIR.mkdir(exist_ok=True)

SEGMENT_TOKEN_SIZE = 1500          # tokens per segment (dissertation §3.3 / §3.5)
MAX_SEGMENTS: Optional[int] = 5    # cap on segments evaluated; set to None to score the whole paper
MAX_RETRIES = 1                    # retries on malformed LLM JSON before giving up on a segment

co = cohere.ClientV2(api_key=COHERE_API_KEY)

## Text extraction & segmentation

In [ ]:
def extract_text_from_pdf(file_bytes: bytes) -> str:
    """Extract concatenated text from every page of a PDF."""
    reader = PdfReader(io.BytesIO(file_bytes))
    return " ".join(page.extract_text() for page in reader.pages if page.extract_text())


def extract_text_from_txt(file_bytes: bytes) -> str:
    """Decode a plain text file, tolerating stray non-UTF-8 bytes."""
    return file_bytes.decode("utf-8", errors="ignore")


def segment_tokens(tokens: list[str], segment_size: int = SEGMENT_TOKEN_SIZE) -> list[str]:
    """Split a token list into space-joined segments of up to `segment_size` tokens."""
    return [" ".join(tokens[i:i + segment_size]) for i in range(0, len(tokens), segment_size)]

## LLM scoring

Same strict REF-reviewer system prompt as v3 (dissertation Appendix C documents the prompt's evolution). The only behavioural change: a malformed/unparseable JSON response is now retried once before falling back, and a failed segment is excluded from the average rather than silently injected as zeros, which previously dragged the calibrated score down in a way that didn't reflect the paper.

In [ ]:
REF_SYSTEM_PROMPT = {
    "role": "system",
    "content": (
        "You are a REF (Research Excellence Framework) reviewer. Be very critical. "
        "Rate the following text realistically from 1–4 on: Originality, Significance, Rigour.\n"
        "- 4 = Truly world-leading and transformative.\n"
        "- 3 = Strong and internationally excellent but not ground-breaking.\n"
        "- 2 = Competent, sound, but limited originality or rigour.\n"
        "- 1 = Weak, flawed, or of national interest only.\n"
        "Avoid inflating scores. Respond with JSON only, no other text: "
        '{"originality": X, "significance": Y, "rigour": Z}'
    ),
}


def _parse_llm_json(reply: str) -> dict:
    """Extract the first {...} block from a reply and parse it as JSON."""
    start, end = reply.find("{"), reply.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError(f"No JSON object found in reply: {reply!r}")
    return json.loads(reply[start:end])


def evaluate_segment_with_llm(segment: str, max_retries: int = MAX_RETRIES) -> Optional[dict]:
    """Score one segment on originality/significance/rigour via the Cohere chat API.

    Returns None (rather than a zeroed dict) if scoring fails after retries, so the
    caller can exclude it from averages instead of treating it as a genuine 0.
    """
    user_prompt = {"role": "user", "content": segment[:2000]}
    last_error: Optional[Exception] = None

    for attempt in range(max_retries + 1):
        try:
            response = co.chat(
                model=MODEL_NAME,
                messages=[REF_SYSTEM_PROMPT, user_prompt],
                temperature=0.1,
            )
            reply = response.message.content[0].text.strip()
            data = _parse_llm_json(reply)
            return {
                "originality": int(data.get("originality", 0)),
                "significance": int(data.get("significance", 0)),
                "rigour": int(data.get("rigour", 0)),
            }
        except Exception as e:  # malformed JSON, API error, etc.
            last_error = e
            continue

    print(f"⚠️ Segment evaluation failed after {max_retries + 1} attempt(s): {last_error}")
    return None

## Calibration & impact weighting

Unchanged from v3 / the dissertation's documented design (§3.3, §3.5): the penalty for `rigour < 3` is deliberate, since rigour was found to be the most predictive dimension of expert assessment in your testing sample — this is a design choice, not a bug, so it's preserved as-is.

In [ ]:
def calibrate_scores(results: list[dict]) -> float:
    """Compress over-generous or internally inconsistent LLM scores into a single 1-4 figure.

    `results` should already have any failed (None) segments filtered out by the caller.
    """
    if not results:
        raise ValueError("calibrate_scores called with no valid results")

    calibrated = []
    for r in results:
        avg = (r["originality"] + r["significance"] + r["rigour"]) / 3
        penalty = 0.0

        # Penalise inconsistency or inflated combinations
        if min(r.values()) <= 2 and max(r.values()) >= 4:
            penalty += 0.5
        if avg > 3.5:
            penalty += 0.3  # overgenerosity clamp
        if r["rigour"] < 3:
            penalty += 0.2  # rigour undervalued → drop total (dissertation §3.5)

        calibrated.append(max(1.0, avg - penalty))

    return sum(calibrated) / len(calibrated)


IMPACT_KEYWORDS = [
    "policy", "public health", "clinical", "societal", "industry", "application",
    "impact", "real-world", "translation", "implementation", "sustainability",
    "biotech", "education", "environment",
]


def assess_impact(text: str) -> tuple[str, float]:
    """Heuristic impact weighting based on keyword frequency. Returns (label, multiplier)."""
    matches = sum(
        bool(re.search(rf"\b{re.escape(kw)}\b", text, flags=re.IGNORECASE))
        for kw in IMPACT_KEYWORDS
    )
    if matches > 15:
        return "High Impact", 1.15
    elif matches > 8:
        return "Medium Impact", 1.10
    elif matches > 3:
        return "Low Impact", 1.05
    return "Minimal Impact", 1.00


def compute_ref_score(calibrated_score: float, impact_factor: float) -> str:
    """Map a calibrated score + impact factor to a REF band label, per dissertation §3.3."""
    weighted_score = min(calibrated_score * impact_factor, 4.0)

    bands = [
        (3.8, "High 4* (World Leading)"),
        (3.5, "Middle 4* (World Leading)"),
        (3.2, "Low 4* (Borderline World Leading)"),
        (2.8, "High 3* (Internationally Excellent)"),
        (2.5, "Middle 3* (Internationally Excellent)"),
        (2.2, "Low 3* (Borderline Excellent)"),
        (1.8, "High 2* (Recognised Internationally)"),
        (1.5, "Middle 2* (Recognised Internationally)"),
        (1.2, "Low 2* (Nationally Recognised)"),
    ]
    for threshold, label in bands:
        if weighted_score >= threshold:
            return f"{label}, {weighted_score:.2f}"
    return f"1* (Below National Standard, {weighted_score:.2f})"

## Caching

Cache key now combines the file hash with the model name and prompt version, so changing `MODEL_NAME` or editing the system prompt (and bumping `PROMPT_VERSION`) automatically invalidates stale cached results instead of silently reusing them.

In [ ]:
def get_cache_key(file_bytes: bytes) -> str:
    file_hash = hashlib.sha256(file_bytes).hexdigest()
    config_fingerprint = hashlib.sha256(f"{MODEL_NAME}:{PROMPT_VERSION}".encode()).hexdigest()[:12]
    return f"{file_hash}_{config_fingerprint}"


def cache_path(cache_key: str) -> Path:
    return CACHE_DIR / f"{cache_key}.pkl"


def save_cache(cache_key: str, data: dict) -> None:
    with open(cache_path(cache_key), "wb") as f:
        pickle.dump(data, f)


def load_cache(cache_key: str) -> Optional[dict]:
    path = cache_path(cache_key)
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    return None


def clear_cache() -> None:
    if CACHE_DIR.exists():
        shutil.rmtree(CACHE_DIR)
    CACHE_DIR.mkdir(exist_ok=True)

## Pipeline

Pulls the steps above together. Separated from the widget callback below so it can be unit-tested or called from a script without touching the UI.

In [ ]:
@dataclass
class EvaluationResult:
    file_name: str
    word_count: int
    segment_count: int
    segments_evaluated: int
    truncated: bool
    failed_segments: int
    impact_level: str
    impact_factor: float
    final_ref: str
    segment_scores: list = field(default_factory=list)


def run_evaluation(file_name: str, file_bytes: bytes, log=print) -> EvaluationResult:
    """Run the full extraction → scoring → calibration → banding pipeline for one file.

    `log` defaults to print but can be swapped for a widget-aware logger in the UI.
    """
    text = (
        extract_text_from_pdf(file_bytes)
        if file_name.lower().endswith(".pdf")
        else extract_text_from_txt(file_bytes)
    )
    if not text.strip():
        raise ValueError(
            "No extractable text found. The PDF may be a scanned image without OCR, "
            "or the file may be corrupted."
        )

    tokens = word_tokenize(text)
    log(f"🧮 Word Count: {len(tokens)}")

    impact_level, impact_factor = assess_impact(text)
    log(f"🌍 Impact Assessment: {impact_level} (x{impact_factor})")

    all_segments = segment_tokens(tokens)
    segments_to_run = all_segments if MAX_SEGMENTS is None else all_segments[:MAX_SEGMENTS]
    truncated = MAX_SEGMENTS is not None and len(all_segments) > MAX_SEGMENTS

    if truncated:
        skipped = len(all_segments) - MAX_SEGMENTS
        log(
            f"⚠️ This paper has {len(all_segments)} segments; only the first "
            f"{MAX_SEGMENTS} are being scored ({skipped} segment(s) / roughly "
            f"{skipped * SEGMENT_TOKEN_SIZE} tokens skipped). Raise MAX_SEGMENTS in the "
            "Config cell to score the whole paper."
        )

    results = []
    failed = 0
    for i, seg in enumerate(segments_to_run):
        log(f"\n➡️ Evaluating segment {i + 1}/{len(segments_to_run)}...")
        res = evaluate_segment_with_llm(seg)
        if res is None:
            failed += 1
            continue
        log(f"   Segment {i + 1}: {res}")
        results.append(res)

    if not results:
        raise RuntimeError(
            "All segments failed to score — check your API key, network connection, "
            "and Cohere account status."
        )
    if failed:
        log(f"⚠️ {failed} segment(s) failed to score and were excluded from the average.")

    calibrated_score = calibrate_scores(results)
    final_ref = compute_ref_score(calibrated_score, impact_factor)
    log(f"\n✅ Final REF Rating: {final_ref}")

    return EvaluationResult(
        file_name=file_name,
        word_count=len(tokens),
        segment_count=len(all_segments),
        segments_evaluated=len(results),
        truncated=truncated,
        failed_segments=failed,
        impact_level=impact_level,
        impact_factor=impact_factor,
        final_ref=final_ref,
        segment_scores=results,
    )

## Interface

Supports multiple file uploads in one go (v3 only supported a single file). Each file is processed and reported independently.

In [ ]:
upload_widget = widgets.FileUpload(accept=".pdf,.txt", multiple=True)
clear_cache_button = widgets.Button(description="🧹 Clear Cache", button_style="warning")
output = widgets.Output()


def _on_clear_cache(_):
    output.clear_output()
    with output:
        clear_cache()
        print("🧹 Cache cleared successfully.")


def _on_upload(change):
    output.clear_output()
    with output:
        if not upload_widget.value:
            print("Please upload at least one file.")
            return

        uploaded_files = (
            upload_widget.value.values()
            if isinstance(upload_widget.value, dict)
            else upload_widget.value
        )

        for uploaded_file in uploaded_files:
            file_name = uploaded_file["name"] if isinstance(uploaded_file, dict) else uploaded_file.name
            file_bytes = uploaded_file["content"] if isinstance(uploaded_file, dict) else uploaded_file.content
            if not isinstance(file_bytes, (bytes, bytearray)):
                file_bytes = bytes(file_bytes)

            print(f"📄 Evaluating paper: {file_name}")

            cache_key = get_cache_key(file_bytes)
            cached = load_cache(cache_key)
            if cached:
                print("✅ Loaded cached result (same file + same model/prompt version).\n")
                print("Final REF Rating:", cached["final_ref"])
                print("-" * 60)
                continue

            try:
                result = run_evaluation(file_name, file_bytes, log=print)
                save_cache(cache_key, {
                    "final_ref": result.final_ref,
                    "segments": result.segment_scores,
                })
                print("💾 Cached result saved.")
            except Exception as e:
                print(f"❌ Evaluation failed for {file_name}: {e}")

            print("-" * 60)


display(widgets.HTML("<h3>📂 Upload academic paper(s) (.pdf or .txt):</h3>"))
display(upload_widget, clear_cache_button, output)
upload_widget.observe(_on_upload, names="value")
clear_cache_button.on_click(_on_clear_cache)